In [1]:
#V3 시총, BOE 통한 GISC기준 산업군별 대표 주 기준정보 불러기기
import pandas as pd
df_sc = pd.read_csv('industry_scode_v3.csv') 
df_sc.head()

,Industry,Keyword,Code
0,바이오,유한양행,000100
1,바이오,삼성바이오,207940
2,바이오,한미약품,128940
3,바이오,대웅제약,069620
4,바이오,셀트리온,068270


In [3]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from time import sleep

def get_foreign_trading(code: str, name, industry, start_year: int = 2025, max_pages: int = 20):
    base_url = f"https://finance.naver.com/item/frgn.naver?code={code}&page="
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_data = []

    for page in range(1, max_pages + 1):
        url = base_url + str(page)
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            print(f"❌ 요청 실패: {url}")
            break

        soup = BeautifulSoup(res.text, 'lxml')
        tables = soup.select("table.type2")

        if len(tables) < 2:
            print(f"⚠️ [{code}] 테이블 수 부족 (page {page}) → 건너뜀")
            continue
        
        table = tables[1]  # 두 번째 테이블: 외국인 매매 내역
        

        rows = table.select("tr")[2:]  # 헤더 제외
        stop_flag = False
        for row in rows:
            cols = row.find_all("td")

            if len(cols) != 9:
                continue
            date = cols[0].text.strip()
                     
            if not date or '.' not in date:
                continue
            year = int(date.split('.')[0])
            if year < start_year:
                stop_flag = True
                break

            diff = re.sub(r'[\n\t,]', '', cols[2].text).strip()
            
            all_data.append({
                "date": date.replace('.', '-'),
                "close": cols[1].text.strip().replace(',', ''),
                "change": diff,
                "change_ratio": cols[3].text.strip().replace(',', ''),
                "volume": cols[4].text.strip().replace(',', ''),
                "institution_vol": cols[5].text.strip().replace(',', ''),
                "foreign_vol": cols[6].text.strip().replace(',', ''),
                "foreign_num": cols[7].text.strip().replace(',', ''),
                "foreign_ratio": cols[8].text
            })

        if stop_flag:
            print(f"✅ {start_year}년 이전 데이터 도달 → 중단")
            break

        print(f"📄 페이지 {page} 수집 완료")
        sleep(1.0)

    df = pd.DataFrame(all_data)
    #df['날짜'] = pd.to_datetime(df['날짜'])
    #df = df.sort_values('날짜')

    df['keyword'] = name
    df['industry'] = industry
    df['ticker'] = code
    
    return df


In [ ]:
df_sc = pd.read_csv('industry_scode_v3.csv') 
df_sc.head()

result_df = pd.DataFrame()

for _, row in df_sc.iterrows():
    code = row['Code']  # 종목코드, 코스피, 미국주식 티커
    name = row['Keyword']  # 종목 키워드
    industry = row['Industry']    
    
    df = get_foreign_trading(code, name, industry, start_year=2025)
    
    #df = fill_weekend_data(df)
    
    result_df = pd.concat([result_df, df], ignore_index=True)

    print(f'산업군:{industry}, 키워드:{name}, 티커:{code}')

result_df.to_csv("foreign_amt_v3_2025.csv", index=False, encoding='utf-8-sig')
print("저장 완료: foreign_amt_v3_2025.csv")